### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots aren’t “talking” in the way humans do— they’re amazing vocal mimics. Their ability to reproduce human speech (and other sounds) comes from a mix of anatomy, social instincts, and evolution. Here’s a breakdown of why many parrots can learn to “talk”:\n\n| Aspect | What it is | How it helps parrots mimic speech |\n|--------|------------|-----------------------------------|\n| **Vocal anatomy** | **Syrinx** – the bird’s sound‑producing organ located at the base of the trachea, plus a highly flexible **tongue, beak, and throat muscles**. | The syrinx can produce a huge variety of tones and pitches, far more than a human larynx. Parrots can finely control airflow and tension, allowing them to copy the subtle nuances of human words. |\n| **Brain structure** | Enlarged **caudomedial nidopallium (NCM)** and **song system** (similar to songbirds). | These brain regions are specialized for auditory learning and memory. They let parrots store and later reproduce complex

In [4]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [5]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'We need to get weather for Boston using function. Use get_weather.', 'tool_calls': [{'id': 'fc_b4096b62-b838-4de0-9074-1e1cbdc6821d', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 127, 'total_tokens': 169, 'completion_time': 0.088037916, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.028381249, 'prompt_tokens_details': None, 'queue_time': 0.246374044, 'total_time': 0.116419165}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c15aa9c1b7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a13662-bb21-7143-8ce2-94945f8ac417-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_b4096b62-b838-4de0-9074-1e1cbdc6821d', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'inp

### Tool Execution Loops

In [6]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston right now is sunny. Enjoy the clear skies!


In [7]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks weather in Boston. Use function get_weather.', 'tool_calls': [{'id': 'fc_8e10df40-3978-4fdd-a81a-5d1ff0ec225b', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 126, 'total_tokens': 165, 'completion_time': 0.081828304, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.014917989, 'prompt_tokens_details': None, 'queue_time': 0.357579673, 'total_time': 0.096746293}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4200b3f836', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1366b-eb57-7702-a099-5e399385193d-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_8e10df40-3978-4fdd-a81a-5d1ff0ec225b'